# Streamlit App - Generative Image Suite


Aplikasi web sederhana pake Streamlit buat text-to-image, inpainting, sama outpainting.\n---


## 1. Install


In [1]:
!pip install streamlit diffusers transformers accelerate torch pillow numpy streamlit-drawable-canvas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.0 MB/s eta 0:00:00


## 2. Kode Aplikasi


jalanin cell ini buat generate file app.py, terus jalanin `streamlit run app.py` di terminal.


In [2]:
%%writefile app.py
import streamlit as st
import torch
import gc
import numpy as np
from PIL import Image, ImageDraw, ImageOps
import io

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
)

st.set_page_config(page_title='Generative Image Suite', page_icon=':art:', layout='wide')

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
INPAINT_MODEL = 'runwayml/stable-diffusion-inpainting'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@st.cache_resource
def load_models():
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False
    ).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)

    pipe_i = StableDiffusionInpaintPipeline.from_pretrained(
        INPAINT_MODEL, torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False
    ).to(DEVICE)
    pipe_i.set_progress_bar_config(disable=True)
    return pipe, pipe_i

def ganti_scheduler(pipe, name):
    if name == 'Euler A':
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif name == 'DPM++':
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    elif name == 'DDIM':
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    return pipe

def bersihin_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    st.success('Memory cleared!')

def generate(pipe, prompt, neg, seed, gs, steps, n):
    hasil = []
    for i in range(n):
        gen = torch.Generator(device=DEVICE).manual_seed(seed + i)
        out = pipe(
            prompt=prompt, negative_prompt=neg,
            generator=gen, guidance_scale=gs, num_inference_steps=steps
        )
        hasil.append(out.images[0])
    return hasil

def buat_grid(images, cols=2):
    rows = (len(images) + cols - 1) // cols
    w, h = images[0].size
    grid = Image.new('RGB', (cols * w, rows * h))
    for i, img in enumerate(images):
        r, c = i // cols, i % cols
        grid.paste(img.resize((w, h)), (c * w, r * h))
    return grid

# === UI ====
st.title('Generative Image Suite')
st.markdown('Aplikasi buat generate dan edit gambar pake Stable Diffusion 1.5')
st.markdown('---')

with st.spinner('Loading models... (butuh beberapa menit)'):
    pipe_t2i, pipe_inpaint = load_models()
st.success('Models ready!')

tab1, tab2 = st.tabs(['Text-to-Image', 'Inpainting / Outpainting'])

# === TAB 1: TEXT-TO-IMAGE ===
with tab1:
    col1, col2 = st.columns([1, 2])
    with col1:
        st.subheader('Settings')
        prompt = st.text_area('Prompt',
            'An astronaut standing on the surface of the Moon, wearing a spacesuit, Earth in the background, stars, detailed, cinematic lighting',
            height=80)
        neg = st.text_area('Negative Prompt',
            'photorealistic, realistic, photograph, 3d render, messy, blurry, low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration',
            height=80)

        col_s1, col_s2 = st.columns(2)
        with col_s1:
            gs = st.slider('Guidance Scale', 1.0, 20.0, 7.5, 0.5)
        with col_s2:
            steps = st.slider('Inference Steps', 5, 100, 30, 5)

        seed = st.number_input('Seed', value=222, step=1)
        num = st.number_input('Jumlah Gambar', min_value=1, max_value=4, value=1)
        sched = st.selectbox('Scheduler', ['Euler A', 'DPM++', 'DDIM'])

        col_b1, col_b2 = st.columns(2)
        with col_b1:
            btn_gen = st.button('Generate', type='primary', use_container_width=True)
        with col_b2:
            if st.button('Clear Memory', use_container_width=True):
                bersihin_memory()

    with col2:
        st.subheader('Hasil')
        if btn_gen:
            with st.spinner('Generating...'):
                pipe_t2i = ganti_scheduler(pipe_t2i, sched)
                images = generate(pipe_t2i, prompt, neg, int(seed), gs, steps, int(num))
                if len(images) == 1:
                    st.image(images[0], use_container_width=True)
                else:
                    st.image(buat_grid(images, 2), use_container_width=True)
                st.session_state['gen_images'] = images
                st.session_state['last_prompt'] = prompt

# === TAB 2: INPAINTING / OUTPAINTING ===
with tab2:
    st.subheader('Edit Gambar')

    if 'gen_images' in st.session_state and len(st.session_state['gen_images']) > 0:
        opts = [f'Image {i+1}' for i in range(len(st.session_state['gen_images']))]
        idx = st.selectbox('Pilih gambar', range(len(opts)), format_func=lambda x: opts[x])
        src = st.session_state['gen_images'][idx].resize((512, 512))
    else:
        st.warning('Generate dulu di tab Text-to-Image atau upload gambar')
        up = st.file_uploader('Upload', type=['png', 'jpg'])
        if up:
            src = Image.open(up).convert('RGB').resize((512, 512))
            st.session_state['gen_images'] = [src]
            idx = 0
        else:
            st.stop()

    col_e1, col_e2 = st.columns([1, 1])
    with col_e1:
        st.image(src, caption='Source', use_container_width=True)
        st.subheader('Inpainting')
        inp_prompt = st.text_input('Prompt inpainting', 'A broken satellite in space, damaged, detailed')

        try:
            from streamlit_drawable_canvas import st_canvas
            canvas = st_canvas(
                fill_color='rgba(255,255,255,1.0)',
                stroke_width=25, stroke_color='rgba(255,255,255,1.0)',
                background_image=src, height=512, width=512,
                drawing_mode='freedraw', key='canvas'
            )
            col_btn, col_clr = st.columns(2)
            with col_btn:
                btn_inp = st.button('Run Inpainting', type='primary', use_container_width=True)
            with col_clr:
                if st.button('Clear Memory', use_container_width=True):
                    bersihin_memory()
        except:
            st.info('Drawable canvas unavailable, pake mode rectangle')
            mx = st.slider('X', 0, 512, 350)
            my = st.slider('Y', 0, 512, 60)
            mw = st.slider('Width', 10, 200, 110)
            mh = st.slider('Height', 10, 200, 140)
            btn_inp = st.button('Run Inpainting', type='primary')
            canvas = None

    with col_e2:
        st.subheader('Result')
        if btn_inp:
            with st.spinner('Inpainting...'):
                if canvas is not None and canvas.image_data is not None:
                    arr = canvas.image_data[:, :, 0].astype(np.uint8)
                    mask_img = Image.fromarray(((arr > 0) * 255).astype(np.uint8), mode='L')
                else:
                    mask_img = Image.new('L', (512, 512), 0)
                    d = ImageDraw.Draw(mask_img)
                    d.rectangle([mx, my, mx+mw, my+mh], fill=255)

                gen = torch.Generator(device=DEVICE).manual_seed(9)
                hasil = pipe_inpaint(
                    prompt=inp_prompt,
                    image=src.resize((512, 512)),
                    mask_image=mask_img.resize((512, 512)),
                    generator=gen, num_inference_steps=30, guidance_scale=7.5
                ).images[0]
                st.image(hasil, use_container_width=True)
                st.session_state['inpainted'] = hasil

        # Zoom Out
        st.markdown('---')
        st.subheader('Zoom-Out')
        if 'inpainted' in st.session_state:
            base_zoom = st.session_state['inpainted']
        elif 'gen_images' in st.session_state:
            base_zoom = st.session_state['gen_images'][0]
        else:
            base_zoom = src

        expand = st.slider('Expand (px)', 32, 256, 96, 16)
        if st.button('Run Zoom-Out', type='primary'):
            with st.spinner('Zooming out...'):
                result = base_zoom.resize((384, 384))
                for direction in ['left', 'right', 'top', 'bottom']:
                    w, h = result.size
                    e = int(expand)
                    if direction == 'right':
                        new_img = Image.new('RGB', (w+e, h), (0,0,0))
                        new_img.paste(result, (0,0))
                        m = Image.new('L', (w+e, h), 0)
                        ImageDraw.Draw(m).rectangle([w, 0, w+e, h], fill=255)
                    elif direction == 'left':
                        new_img = Image.new('RGB', (w+e, h), (0,0,0))
                        new_img.paste(result, (e,0))
                        m = Image.new('L', (w+e, h), 0)
                        ImageDraw.Draw(m).rectangle([0, 0, e, h], fill=255)
                    elif direction == 'top':
                        new_img = Image.new('RGB', (w, h+e), (0,0,0))
                        new_img.paste(result, (0,e))
                        m = Image.new('L', (w, h+e), 0)
                        ImageDraw.Draw(m).rectangle([0, 0, w, e], fill=255)
                    else:
                        new_img = Image.new('RGB', (w, h+e), (0,0,0))
                        new_img.paste(result, (0,0))
                        m = Image.new('L', (w, h+e), 0)
                        ImageDraw.Draw(m).rectangle([0, h, w, h+e], fill=255)

                    prompt_zoom = st.session_state.get('last_prompt', 'moon surface, stars')
                    result = pipe_inpaint(
                        prompt=prompt_zoom, image=new_img, mask_image=m,
                        generator=torch.Generator(device=DEVICE).manual_seed(42),
                        num_inference_steps=30, guidance_scale=7.5
                    ).images[0]
                st.image(result, use_container_width=True)

st.markdown('---')
st.caption('Generative Image Suite - BFGAI Submission')


Writing app.py


## 3. Jalankan Streamlit


Setelah cell di atas di-run, bakal ada file `app.py`.\nBuka terminal dan jalanin:\n```bash\nstreamlit run app.py\n```
